# 01 · Feature analysis
Do the six MVP features separate ring accounts from legitimate accounts (including the lookalikes: shopkeeper, gig worker, refunds, high-volume business)?

_Synthetic data only._

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "backend"))   # run from repo root or notebooks/
import pandas as pd
import matplotlib.pyplot as plt
from app.simulator import generate_dataset
from app.detection.features import FEATURE_NAMES, compute_features_batch
from app.detection.scorer import RiskScorer


In [ ]:
data = generate_dataset(seed=42)
txs, truth = data["transactions"], {a for m in data["ring_meta"] for a in m["members"]}
feats = compute_features_batch(txs)
df = pd.DataFrame.from_dict(feats, orient="index")
df["is_ring"] = df.index.isin(truth)
lookalikes = ["acct_shop_001","acct_gig_001","acct_biz_001","acct_refund_merchant","acct_student_01"]
print(len(df), "accounts;", int(df.is_ring.sum()), "ring accounts")
df.groupby("is_ring")[list(FEATURE_NAMES)].median().T

## Distribution by class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, name in zip(axes.ravel(), FEATURE_NAMES):
    clip = df[name].clip(upper=df[name].quantile(0.98))
    for flag, label in ((False, "other"), (True, "ring")):
        ax.hist(clip[df.is_ring == flag], bins=25, alpha=0.6, label=label, density=True)
    ax.set_title(name)
axes[0,0].legend(); plt.tight_layout(); plt.show()

## Legitimate lookalikes — what do they look like to the features?

In [ ]:
scorer = RiskScorer()
rows = []
for a in lookalikes:
    if a in feats:
        r = scorer.score(feats[a]); rows.append({"account": a, "risk": r.risk_score, "level": r.risk_level.value, "why": "; ".join(r.reasons[:2])})
pd.DataFrame(rows)

**Next features to add (plan §6.2):** sender novelty, burstiness, round-trip ratio, cover activity, new-beneficiary ratio. Add them to `detection/features.py::FEATURE_NAMES` and extend `scorer._WEIGHTS/_SAT`.